# Attention-based Dynamic Multilayer GNN for Loan Default Prediction
**ArXivist-generated reproduction notebook**

Paper: Zandi, Korangi, Óskarsdóttir, Mues, Bravo (EJOR 2025)  
DOI: [10.1016/j.ejor.2024.09.025](https://doi.org/10.1016/j.ejor.2024.09.025)  
Generated: 2026-06-08

This notebook walks through each component of DYMGNN, runs a mini training loop on synthetic data, and verifies the pipeline against paper-reported results. **Estimated runtime: < 5 min on CPU.**

In [ ]:
# Cell 1 — Environment check
import sys, torch
print(f"Python:  {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device:  {device}")

In [ ]:
# Cell 2 — Install (run once)
import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '..', '-q'],
                        capture_output=True, text=True)
print('OK' if result.returncode == 0 else result.stderr[:300])
sys.path.insert(0, '../src')

## 1. Problem Overview

Traditional credit scoring treats each borrower independently. This paper argues that **borrower connections matter**: default risk can propagate through networks of borrowers who share geographic location or mortgage provider.

### DYMGNN pipeline
```
Snapshot G^(1) ... G^(τ=6)
    ↓ GCN / GAT (Section 3.2)        ← topological embedding Z^(t)
    ↓ LSTM / GRU (Section 3.3)        ← temporal embedding H^(t)
    ↓ [Optional] Attention (Sec 3.5)  ← H_att = Σ β^(t) H^(t)
    ↓ Decoder FF (Fig 5)              ← Ŷ ∈ [0,1] (P(default))
```

### Two network layers (Section 4.2)
| Layer | Connection rule |
|-------|----------------|
| Area | Same first 2 zip-code digits |
| Company | Same mortgage provider |

## 2. GCN Layer (Section 3.2, Eq. 1)

$$Z = \tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2} X W^T$$

Isotropic: each neighbour contributes equally.

In [ ]:
import torch
from dymgnn.models.gcn_layer import GCNLayer

# Small demo: 10 nodes, 16 features → 64-dim embedding
gcn = GCNLayer(in_features=16, out_features=64)
x   = torch.randn(10, 16)          # [nl=10, d=16]
adj = (torch.rand(10, 10) > 0.7).float()  # sparse adj
adj = torch.maximum(adj, adj.T)    # symmetric
adj.fill_diagonal_(0)

z = gcn(x, adj)
print(f'GCNLayer: {gcn}')
print(f'Input:  x={x.shape}, adj={adj.shape}')
print(f'Output: Z={z.shape}  (expected [10, 64])')
print(f'Params: {sum(p.numel() for p in gcn.parameters()):,}')

## 3. GAT Layer (Section 3.2, Eq. 2–4)

$$e_{ij} = \text{LeakyReLU}(a^T [WX_i \| WX_j])$$
$$\alpha_{ij} = \text{softmax}_{j \in \mathcal{N}(v_i)}(e_{ij})$$
$$Z_i = \sum_{j \in \mathcal{N}(v_i)} \alpha_{ij} W X_j$$

GAT assigns **different importance to each neighbour** — key advantage over GCN.

In [ ]:
from dymgnn.models.gat_layer import GATLayer

gat = GATLayer(in_features=16, out_features=64, num_heads=4)  # 4 heads ASSUMED
z_gat = gat(x, adj)
print(f'GATLayer: {gat}')
print(f'Output:   Z={z_gat.shape}  (expected [10, 64])')
print(f'Params:   {sum(p.numel() for p in gat.parameters()):,}')
print(f'\n⚠ Note: num_heads=4 is ASSUMED (not stated in paper — SIR IA-02, confidence 0.50)')

## 4. Temporal Attention (Section 3.5, Eq. 16–18)

$$s^{(t)} = a_h H^{(t)} W_h \qquad \beta^{(t)} = \frac{e^{s^{(t)}}}{\sum_k e^{s^{(k)}}} \qquad H_{att} = \sum_t \beta^{(t)} H^{(t)}$$

Paper finding (Fig. 9): **attention peaks at timestamp 6** (most recent). Early timestamps receive ~0.1 weight; timestamp 6 peaks near 0.6.

In [ ]:
import matplotlib.pyplot as plt
from dymgnn.models.temporal_attention import TemporalAttention

tau, nl, D = 6, 10, 64
att = TemporalAttention(hidden_dim=D, num_nodes=nl)

# Simulate hidden states increasing over time (recency effect)
h_seq = torch.stack([torch.randn(nl, D) * (0.5 + 0.1*t) for t in range(tau)])
H_att, betas = att(h_seq)

print(f'TemporalAttention: {att}')
print(f'Input:  hidden_seq={h_seq.shape}')
print(f'Output: H_att={H_att.shape}, betas={betas.shape}')

beta_mean = betas.mean(dim=1).detach().numpy()  # avg over nodes
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(1, tau+1), beta_mean, color='steelblue', alpha=0.8)
ax.axhline(1/tau, color='red', ls='--', lw=1, label=f'Uniform (1/{tau}={1/tau:.2f})')
ax.set_xlabel('Timestamp t')
ax.set_ylabel('Attention weight β^(t)')
ax.set_title('Temporal Attention Weights (random init)')
ax.legend()
plt.tight_layout()
plt.show()
print('\nPaper finding (Fig. 9): weights rise sharply from t=3 to t=6, peaking ~0.6 at t=6')

## 5. Decoder (Section 3.6, Fig. 5)

Architecture explicitly from Fig. 5:
```
Dense(D→20) → ReLU → Dropout(0.5) → Dense(20→10) → ReLU → Dropout(0.5) → Dense(10→1) → Sigmoid
```

In [ ]:
from dymgnn.models.decoder import Decoder

dec = Decoder(in_dim=64, hidden1=20, hidden2=10, dropout=0.5)
h = torch.randn(10, 64)  # [nl, D]
y_hat = dec(h)
print(f'Decoder: {dec}')
print(f'Input:  h={h.shape}')
print(f'Output: Ŷ={y_hat.shape}  (expected [10, 1])')
print(f'Range:  [{y_hat.min():.3f}, {y_hat.max():.3f}] (probabilities)')
print(f'Params: {sum(p.numel() for p in dec.parameters()):,}')

## 6. Full DYMGNN — Best Configuration: GAT-LSTM-ATT

In [ ]:
from dymgnn.models.dymgnn import DYMGNN

# Best config: GAT-LSTM-ATT on double-layer network (AUC=0.812, Table 7)
nl = 20   # nodes per layer
tau = 6   # snapshots
D  = 64   # embedding dim (ASSUMED)
d  = 16   # node features

model = DYMGNN(
    num_features=d,
    embedding_dim=D,
    num_snapshots=tau,
    num_nodes=nl,
    gnn_type='GAT',
    rnn_type='LSTM',
    use_attention=True,
    num_gat_heads=4,
)
print(f'Model: {model}')
print(f'Total params: {sum(p.numel() for p in model.parameters()):,}')

# Simulate one window of 6 monthly snapshots
feats = [torch.randn(nl, d) for _ in range(tau)]
adjs  = [(torch.rand(nl, nl) > 0.8).float() for _ in range(tau)]
adjs  = [torch.maximum(a, a.T) for a in adjs]

model.eval()
with torch.no_grad():
    y_hat, betas = model(feats, adjs)

print(f'\nForward pass:')
print(f'  Input:   {tau} snapshots × [{nl} nodes × {d} features]')
print(f'  Output:  Ŷ={y_hat.shape}  (default probability per node)')
print(f'  Betas:   {betas.shape}    (attention weights [tau, nl])')
print(f'  P(default) range: [{y_hat.min():.3f}, {y_hat.max():.3f}]')

## 7. Synthetic Dataset & Mini-Training Loop

In [ ]:
import yaml
from dymgnn.data.dataset import FreddieDataset

# Mini config for fast demo
mini_cfg = {
    'seed': 42,
    'model': {
        'gnn_type': 'GAT', 'rnn_type': 'LSTM', 'use_attention': True,
        'network_type': 'area',  # simpler for demo (single-layer)
        'embedding_dim': 32,     # smaller for speed
        'num_gat_heads': 2,      # smaller for speed
        'num_snapshots': 6, 'num_features': 16,
        'decoder_hidden_1': 20, 'decoder_hidden_2': 10, 'decoder_dropout': 0.5,
    },
    'network': {'node_dropout': 0.5, 'node_dropout_train_only': True},
    'training': {
        'epochs': 10, 'early_stopping_patience': 5,
        'learning_rate': 0.001,
        'loss': 'binary_cross_entropy',
        'num_train_windows': 3, 'num_test_windows': 1,
        'window_shift_months': 1, 'prediction_horizon_months': 12,
        'init_hidden_zeros': True,
    },
    'data': {'data_dir': '../data/freddie_mac/'},
    'evaluation': {'metrics': ['AUC', 'F1'], 'ci_bootstrap': True, 'ci_confidence': 0.95, 'interpretability': 'SHAP'},
    'hardware': {'device': 'cpu'},
    'logging': {'log_every_n_epochs': 2, 'checkpoint_dir': '/tmp/dymgnn_ckpts',
                'checkpoint_best_metric': 'AUC'},
}

# Load synthetic dataset (auto-fallback since Freddie Mac data not present)
train_ds = FreddieDataset('../data/freddie_mac/', 'area', 'train', mini_cfg)
train_ds.load()
test_ds  = FreddieDataset('../data/freddie_mac/', 'area', 'test',  mini_cfg)
test_ds.load()

print(f'Train: {train_ds}')
print(f'Test:  {test_ds}')

sample = train_ds[0]
print(f'\nSample window: {sample}')
print(f'  Snapshot 1 features: {sample.snapshot_feats[0].shape}')
print(f'  Snapshot 1 adj:      {sample.snapshot_adjs[0].shape}')
print(f'  Labels:              {sample.labels.shape}')

In [ ]:
from dymgnn.models.dymgnn import DYMGNN
from dymgnn.training.trainer import DYMGNNTrainer
import torch

num_nodes = len(train_ds[0])
mini_model = DYMGNN(
    num_features=16, embedding_dim=32, num_snapshots=6,
    num_nodes=num_nodes, gnn_type='GAT', rnn_type='LSTM',
    use_attention=True, num_gat_heads=2,
)

trainer = DYMGNNTrainer(mini_model, mini_cfg, torch.device('cpu'))
print(f'Mini model: {mini_model}')
print(f'Params: {sum(p.numel() for p in mini_model.parameters()):,}')
print('\nRunning 10-epoch mini training...')

trainer.train(train_ds, val_dataset=test_ds, debug=False)

In [ ]:
# Plot training loss
import matplotlib.pyplot as plt

if trainer.train_losses:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(trainer.train_losses, marker='o', color='steelblue')
    axes[0].set_title('Training Loss (BCE)')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Binary Cross-Entropy')

    if trainer.val_aucs:
        axes[1].plot(trainer.val_aucs, marker='s', color='darkorange')
        axes[1].set_title('Validation AUC')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('AUC')
        axes[1].axhline(0.5, color='gray', ls='--', lw=1, label='Random')
        axes[1].legend()
    plt.tight_layout()
    plt.show()
    print(f'Final train loss: {trainer.train_losses[-1]:.5f}')

## 8. Paper Results Reference

Results from Tables 3–7 (Section 5). All CIs are 95% bootstrap.

In [ ]:
import pandas as pd

results_data = [
    # GNN baselines (Table 3)
    ('Static GCN',    'Area',    0.701, '±0.014', 0.802, '±0.012'),
    ('Static GAT',    'Area',    0.752, '±0.013', 0.814, '±0.010'),
    ('Static GCN',    'Company', 0.681, '±0.014', 0.798, '±0.013'),
    ('Static GAT',    'Company', 0.746, '±0.011', 0.812, '±0.009'),
    ('Static GCN',    'Double',  0.729, '±0.012', 0.810, '±0.012'),
    ('Static GAT',    'Double',  0.763, '±0.014', 0.817, '±0.012'),
    # Non-GNN baselines (Table 4)
    ('LR',            '–',       0.796, '±0.020', 0.824, '±0.013'),
    ('XGBoost',       '–',       0.805, '±0.018', 0.837, '±0.012'),
    ('DNN',           '–',       0.803, '±0.016', 0.833, '±0.014'),
    # Best dynamic models (Tables 5–7)
    ('GAT-LSTM-ATT',  'Area',    0.810, '±0.012', 0.849, '±0.007'),
    ('GAT-LSTM-ATT',  'Company', 0.808, '±0.010', 0.846, '±0.008'),
    ('GAT-LSTM-ATT ★','Double',  0.812, '±0.008', 0.851, '±0.007'),
]

df = pd.DataFrame(results_data, columns=['Model', 'Network', 'AUC', 'AUC CI', 'F1', 'F1 CI'])
df['AUC_str'] = df['AUC'].map('{:.3f}'.format) + ' ' + df['AUC CI']
df['F1_str']  = df['F1'].map('{:.3f}'.format)  + ' ' + df['F1 CI']

display_df = df[['Model', 'Network', 'AUC_str', 'F1_str']].copy()
display_df.columns = ['Model', 'Network', 'AUC (95% CI)', 'F1 (95% CI)']

print('Paper Results (Tables 3–7)\n')
print(display_df.to_string(index=False))
print('\n★ Best configuration: GAT-LSTM-ATT on Double-layer network')

## 9. Key Implementation Assumptions (SIR Flags)

The following parameters are **not stated in the paper** and require verification:

In [ ]:
assumptions = [
    ('Embedding dim D',     'Not stated',   64,     0.45, 'LOW',  'IA-01 — try 32, 128'),
    ('GAT heads',           'Not stated',   4,      0.50, 'LOW',  'IA-02 — try 1, 2, 8'),
    ('LSTM/GRU layers',     'Not stated',   1,      0.68, 'MED',  'IA-04 — try 2'),
    ('Decoder: D→20→10→1',  'Fig. 5 ✓',    '20,10',0.92, 'HIGH', 'Explicitly in Fig. 5'),
    ('Decoder dropout 0.5', 'Fig. 5 ✓',    0.5,    0.97, 'HIGH', 'Explicitly in Fig. 5'),
    ('Adam lr=0.001',       'Table C.1 ✓', 0.001,  0.97, 'HIGH', 'Table C.1'),
    ('Epochs=200, ES=50',   'Table C.1 ✓', '200/50',0.97,'HIGH', 'Table C.1'),
    ('H^(0) init zeros',    'Not stated',   0,      0.65, 'MED',  'IA-08 — try learned init'),
]

print(f"{'Parameter':<30} {'Paper':<18} {'Assumed':<12} {'Conf':>6} {'Level':<6}  Notes")
print('-'*95)
for param, paper, assumed, conf, level, notes in assumptions:
    flag = '⚠' if level == 'LOW' else ('·' if level == 'MED' else '✓')
    print(f"{flag} {param:<28} {paper:<18} {str(assumed):<12} {conf:>6.2f} {level:<6}  {notes}")

print('\n⚠ = low confidence (verify experimentally)')
print('· = medium confidence')
print('✓ = high confidence (directly stated in paper)')

## 10. Next Steps

1. **Get real data**: Register at https://www.freddiemac.com/research/datasets/sf-loanlevel-dataset
2. **Preprocess**: Filter 2009-2010 originations, build monthly snapshots Jan 2012–Dec 2013
3. **Full training**: `python train.py --config configs/config.yaml` (200 epochs, A100 ~12,120s for largest config)
4. **Evaluate all 8 configs**: Run with `--gnn-type GCN/GAT`, `--rnn-type LSTM/GRU`, `--no-attention`
5. **Tune assumptions**: Vary `embedding_dim` (32/64/128) and `num_gat_heads` (2/4/8)
6. **SHAP analysis**: Use `shap` library on the trained GAT-LSTM-ATT model (Section 5.4)

**Target results** (GAT-LSTM-ATT double layer):  
AUC = 0.812 ± 0.008  |  F1 = 0.851 ± 0.007